# INE5664 - Aula prática — Árvores de Decisão e Métodos de Ensemble

**Objetivo da aula:** construir uma compreensão prática de árvores de decisão e métodos de ensemble, conectando conceitos teóricos com implementação e experimentos em Python.

Ao final, você deverá conseguir:

1. Calcular medidas de impureza como **entropia**, **índice Gini** e **erro de classificação**.
2. Entender como uma árvore escolhe divisões em atributos categóricos e contínuos.
3. Treinar árvores de classificação e regressão com `scikit-learn`.
4. Investigar **sobreajuste**, **pré-poda** e **pós-poda por custo-complexidade**.
5. Comparar árvores individuais com ensembles: **Bagging**, **Random Forest**, **AdaBoost**, **Gradient Boosting** e **Stacking**.
6. Realizar um mini-projeto experimental com ajuste de hiperparâmetros e avaliação crítica.

> Esta aula foi organizada como uma atividade prática guiada. Algumas células são demonstrações; outras são exercícios.

In [ ]:
# Ambiente base
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from math import log2

from sklearn.datasets import make_moons, make_classification, load_breast_cancer, load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 5)

In [ ]:
def plot_decision_boundary(model, X, y, title="Fronteira de decisão", ax=None, grid_steps=140):
    # Plota a fronteira de decisão de um classificador 2D já treinado.
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 5))

    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, grid_steps),
        np.linspace(y_min, y_max, grid_steps),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25)
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolor="k", s=40)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")
    ax.set_title(title)
    return ax


def evaluate_classifier(model, X_train, X_test, y_train, y_test, name="modelo"):
    # Treina um classificador e retorna acurácias de treino e teste.
    model.fit(X_train, y_train)
    pred_train = model.predict(X_train)
    pred_test = model.predict(X_test)
    return {
        "modelo": name,
        "acc_treino": accuracy_score(y_train, pred_train),
        "acc_teste": accuracy_score(y_test, pred_test),
    }


def safe_adaboost(base_estimator, **kwargs):
    # Compatibiliza diferentes versões do scikit-learn para AdaBoostClassifier.
    try:
        return AdaBoostClassifier(estimator=base_estimator, **kwargs)
    except TypeError:
        return AdaBoostClassifier(base_estimator=base_estimator, **kwargs)

## 1. Medidas de impureza em árvores de classificação

Árvores de decisão constroem regras de divisão buscando tornar os nós resultantes mais puros.
Nesta seção, vamos implementar três medidas comuns:

- **Entropia:** mede incerteza/aleatoriedade da distribuição das classes.
- **Índice Gini:** mede impureza com base nas probabilidades quadráticas das classes.
- **Erro de classificação:** mede a fração de exemplos que seriam classificados incorretamente se o nó predissesse apenas a classe majoritária.

Para um nó com proporções de classes $$(p_1, \ldots, p_m)$$:

$$
H(t) = -\sum_i p_i \log_2(p_i)
$$

$$
Gini(t) = 1 - \sum_i p_i^2
$$

$$
Erro(t) = 1 - \max_i p_i
$$

In [ ]:
def class_probabilities(y):
    counts = Counter(y)
    n = len(y)
    return np.array([count / n for count in counts.values()], dtype=float)


def entropy(y):
    probs = class_probabilities(y)
    probs = probs[probs > 0]
    return -np.sum(probs * np.log2(probs))


def gini(y):
    probs = class_probabilities(y)
    return 1 - np.sum(probs ** 2)


def classification_error(y):
    probs = class_probabilities(y)
    return 1 - np.max(probs)

# Exemplos equivalentes aos casos de classificação binária discutidos em aula
examples = {
    "puro_0_6": [1, 1, 1, 1, 1, 1],
    "quase_puro_1_5": [0, 1, 1, 1, 1, 1],
    "balanceado_3_3": [0, 0, 0, 1, 1, 1],
}

rows = []
for name, y_node in examples.items():
    rows.append({
        "nó": name,
        "entropia": entropy(y_node),
        "gini": gini(y_node),
        "erro": classification_error(y_node),
    })

pd.DataFrame(rows)

In [ ]:
p_values = np.linspace(0, 1, 101)
entropies = []
ginis = []
errors = []

for p in p_values:
    y_fake = np.array([1] * int(round(p * 1000)) + [0] * int(round((1 - p) * 1000)))
    if len(y_fake) == 0:
        entropies.append(0)
        ginis.append(0)
        errors.append(0)
    else:
        entropies.append(entropy(y_fake))
        ginis.append(gini(y_fake))
        errors.append(classification_error(y_fake))

plt.figure(figsize=(7, 5))
plt.plot(p_values, entropies, label="Entropia")
plt.plot(p_values, ginis, label="Gini")
plt.plot(p_values, errors, label="Erro")
plt.xlabel("P(y = 1)")
plt.ylabel("Impureza")
plt.title("Medidas de impureza em classificação binária")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. Exemplo guiado: dataset “Joga?”

Vamos reproduzir computacionalmente o exemplo clássico em que queremos decidir se alguém joga ou não um esporte dadas condições do tempo.

A ideia é calcular o ganho de informação para diferentes atributos e observar qual divisão seria escolhida no primeiro nó da árvore.

In [ ]:
dados_joga = pd.DataFrame({
    "Tempo": ["Chuvoso", "Ensolarado", "Ensolarado", "Nublado", "Chuvoso", "Chuvoso", "Nublado", "Nublado", "Ensolarado", "Chuvoso", "Nublado", "Ensolarado", "Ensolarado", "Chuvoso"],
    "Temperatura": [21.67, 20.56, 26.67, 28.33, 21.11, 18.33, 17.78, 22.22, 23.89, 20.00, 27.22, 29.44, 22.22, 23.89],
    "Umidade": [91, 70, 90, 86, 96, 70, 65, 90, 70, 80, 75, 85, 95, 80],
    "Vento": ["Sim", "Não", "Sim", "Não", "Não", "Sim", "Sim", "Sim", "Sim", "Não", "Não", "Não", "Não", "Não"],
    "Joga": ["Não", "Sim", "Não", "Sim", "Sim", "Não", "Sim", "Sim", "Sim", "Sim", "Sim", "Não", "Não", "Sim"],
})

dados_joga

A célula a seguir irá consolidar os ganhos de informação calculados para cada atributo (`Tempo` e `Vento` como categóricos, `Temperatura` e `Umidade` como contínuos) em uma única tabela. Após preencher a tabela `ig_summary`, será possível identificar qual atributo oferece o maior ganho de informação e, portanto, qual seria o atributo escolhido para a primeira divisão (nó raiz) da árvore de decisão.

In [ ]:
def information_gain_categorical(df, feature, target):
    # Calcula ganho de informação para uma divisão categórica.
    parent_entropy = entropy(df[target].values)
    n = len(df)
    weighted_child_entropy = 0

    for _, subset in df.groupby(feature):
        weight = len(subset) / n
        weighted_child_entropy += weight * entropy(subset[target].values)

    return parent_entropy - weighted_child_entropy


def candidate_thresholds(values):
    # Retorna pontos médios entre valores consecutivos distintos.
    unique_values = np.sort(np.unique(values))
    return (unique_values[:-1] + unique_values[1:]) / 2


def information_gain_continuous(df, feature, target, threshold):
    # Calcula ganho de informação para uma divisão binária feature <= threshold.
    parent_entropy = entropy(df[target].values)
    left = df[df[feature] <= threshold]
    right = df[df[feature] > threshold]

    n = len(df)
    child_entropy = (len(left) / n) * entropy(left[target].values) + (len(right) / n) * entropy(right[target].values)
    return parent_entropy - child_entropy


def best_continuous_split(df, feature, target):
    rows = []
    for threshold in candidate_thresholds(df[feature].values):
        rows.append({
            "atributo": feature,
            "ponto_de_corte": threshold,
            "IG": information_gain_continuous(df, feature, target, threshold),
        })
    return pd.DataFrame(rows).sort_values("IG", ascending=False).reset_index(drop=True)

print("Entropia da classe Joga:", round(entropy(dados_joga["Joga"].values), 3))

ig_tempo = information_gain_categorical(dados_joga, "Tempo", "Joga")
ig_vento = information_gain_categorical(dados_joga, "Vento", "Joga")

print("IG(Tempo):", round(ig_tempo, 3))
print("IG(Vento):", round(ig_vento, 3))

best_temp = best_continuous_split(dados_joga, "Temperatura", "Joga")
best_umid = best_continuous_split(dados_joga, "Umidade", "Joga")

print("Melhor corte de Temperatura:")
display(best_temp.head(3))
print("Melhor corte de Umidade:")
display(best_umid.head(3))

### Exercício 1 — Escolha do primeiro nó

Complete a célula abaixo para criar uma tabela com o melhor ganho de informação de cada atributo:

- `Tempo`: divisão categórica.
- `Vento`: divisão categórica.
- `Temperatura`: melhor divisão contínua.
- `Umidade`: melhor divisão contínua.

Em seguida, identifique qual atributo seria escolhido para o nó raiz.

In [ ]:
# TODO: Complete a tabela `ig_summary` com os ganhos de informação para cada atributo.
#
# Para os atributos categóricos (`Tempo` e `Vento`):
#   - O 'IG' deve vir das variáveis `ig_tempo` e `ig_vento` calculadas anteriormente.
#   - A 'melhor_regra' pode ser uma descrição genérica, como "Divisão Categórica".
#
# Para os atributos contínuos (`Temperatura` e `Umidade`):
#   - O maior 'IG' e o 'ponto_de_corte' correspondente para cada atributo estão na primeira linha
#     (índice 0) dos DataFrames `best_temp` e `best_umid`. Por exemplo:
#     `ig_temperatura = best_temp.loc[0, 'IG']`
#     `ponto_corte_temperatura = best_temp.loc[0, 'ponto_de_corte']`
#   - A 'melhor_regra' deve descrever a divisão usando o ponto de corte, por exemplo,
#     `f"Temperatura <= {ponto_corte_temperatura:.2f}"`. Lembre-se de formatar o valor.

ig_summary = pd.DataFrame([
    # Exemplo de como preencher uma linha (descomente e adapte para cada atributo):
    # {"atributo": "Tempo", "melhor_regra": "Divisão Categórica (Tempo)", "IG": ig_tempo},
    # {"atributo": "...", "melhor_regra": "...", "IG": ...}
])

# As linhas abaixo já estão prontas para exibir e identificar o melhor atributo após o preenchimento.
# display(ig_summary.sort_values(by="IG", ascending=False).reset_index(drop=True))
#
# print("\nO atributo com o maior ganho de informação para o nó raiz é:")
# print(ig_summary.sort_values(by="IG", ascending=False).iloc[0]["atributo"])

## 3. Árvores de decisão com `scikit-learn`

Agora vamos treinar árvores de decisão em um problema sintético bidimensional. Isso nos permite visualizar a fronteira de decisão e analisar o efeito de hiperparâmetros como `max_depth`, `min_samples_leaf` e `criterion`.

In [ ]:
X, y = make_moons(n_samples=400, noise=0.25, random_state=RANDOM_STATE)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

plt.figure(figsize=(7, 5))
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, edgecolor="k", s=35)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("Conjunto make_moons — treino")
plt.show()

In [ ]:
tree_full = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree_shallow = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)

results = []
for name, model in [("Árvore sem restrição", tree_full), ("Árvore max_depth=3", tree_shallow)]:
    results.append(evaluate_classifier(model, X_train, X_test, y_train, y_test, name))

pd.DataFrame(results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(tree_full, X_test, y_test, "Árvore sem restrição", ax=axes[0])
plot_decision_boundary(tree_shallow, X_test, y_test, "Árvore com max_depth=3", ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(16, 8))
plot_tree(
    tree_shallow,
    filled=True,
    rounded=True,
    feature_names=["x1", "x2"],
    class_names=["classe 0", "classe 1"],
)
plt.title("Estrutura da árvore com max_depth=3")
plt.show()

print(export_text(tree_shallow, feature_names=["x1", "x2"]))

## 4. Pré-poda: controlando complexidade antes da árvore crescer

A pré-poda utiliza critérios de parada para limitar a complexidade da árvore. Exemplos:

- `max_depth`: profundidade máxima.
- `min_samples_split`: número mínimo de amostras para dividir um nó.
- `min_samples_leaf`: número mínimo de amostras em uma folha.
- `max_leaf_nodes`: número máximo de folhas.

Vamos observar o efeito de `max_depth`.

In [ ]:
rows = []
for depth in range(1, 16):
    model = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    rows.append({
        "max_depth": depth,
        "acc_treino": accuracy_score(y_train, model.predict(X_train)),
        "acc_teste": accuracy_score(y_test, model.predict(X_test)),
        "n_folhas": model.get_n_leaves(),
    })

depth_results = pd.DataFrame(rows)
depth_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(depth_results["max_depth"], depth_results["acc_treino"], marker="o", label="Treino")
plt.plot(depth_results["max_depth"], depth_results["acc_teste"], marker="o", label="Teste")
plt.xlabel("max_depth")
plt.ylabel("Acurácia")
plt.title("Efeito da profundidade máxima")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Pós-poda por custo-complexidade

Na pós-poda, primeiro permitimos que uma árvore cresça e depois removemos partes da árvore que não melhoram suficientemente a generalização.

No CART, uma estratégia comum é a **poda custo-complexidade**, controlada pelo parâmetro `ccp_alpha`.

In [ ]:
base_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
path = base_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

# O último alpha costuma gerar uma árvore só com a raiz. Vamos removê-lo da comparação.
ccp_alphas = ccp_alphas[:-1]

rows = []
for alpha in ccp_alphas:
    model = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=alpha)
    model.fit(X_train, y_train)
    rows.append({
        "ccp_alpha": alpha,
        "acc_treino": accuracy_score(y_train, model.predict(X_train)),
        "acc_teste": accuracy_score(y_test, model.predict(X_test)),
        "n_folhas": model.get_n_leaves(),
        "profundidade": model.get_depth(),
    })

pruning_results = pd.DataFrame(rows)
pruning_results.sort_values("acc_teste", ascending=False).head(10)

In [ ]:
best_alpha = pruning_results.sort_values("acc_teste", ascending=False).iloc[0]["ccp_alpha"]
pruned_tree = DecisionTreeClassifier(random_state=RANDOM_STATE, ccp_alpha=best_alpha)
pruned_tree.fit(X_train, y_train)

plt.figure(figsize=(8, 5))
plt.plot(pruning_results["ccp_alpha"], pruning_results["acc_treino"], marker="o", label="Treino")
plt.plot(pruning_results["ccp_alpha"], pruning_results["acc_teste"], marker="o", label="Teste")
plt.xscale("symlog", linthresh=1e-5)
plt.xlabel("ccp_alpha")
plt.ylabel("Acurácia")
plt.title("Pós-poda por custo-complexidade")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Melhor ccp_alpha encontrado: {best_alpha:.6f}")
print("Folhas:", pruned_tree.get_n_leaves(), "| Profundidade:", pruned_tree.get_depth())

## 6. Árvores de regressão

Em árvores de regressão, as folhas retornam valores contínuos. Com erro quadrático, a predição de uma folha é tipicamente a média dos valores da variável resposta das observações que caem naquela folha.

Vamos usar o dataset `diabetes` para comparar uma árvore individual com ensembles de regressão.

In [ ]:
diabetes = load_diabetes()
X_reg = diabetes.data
y_reg = diabetes.target

Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.30, random_state=RANDOM_STATE
)

reg_models = {
    "Árvore regressora": DecisionTreeRegressor(random_state=RANDOM_STATE, max_depth=4),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=60, random_state=RANDOM_STATE),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

reg_rows = []
for name, model in reg_models.items():
    model.fit(Xr_train, yr_train)
    pred = model.predict(Xr_test)
    reg_rows.append({
        "modelo": name,
        "RMSE": np.sqrt(mean_squared_error(yr_test, pred)),
        "R2": r2_score(yr_test, pred),
    })

pd.DataFrame(reg_rows).sort_values("RMSE")

# Parte II — Métodos de Ensemble

Métodos de ensemble combinam predições de múltiplos modelos de base. A ideia central é que modelos diferentes podem cometer erros diferentes; ao combinar seus resultados, muitas vezes obtemos melhor generalização do que com um único modelo.

Vamos estudar:

- Votação majoritária.
- Bagging e amostragem bootstrap.
- Random Forest.
- AdaBoost.
- Gradient Boosting.
- Stacking.

## 7. Por que ensembles podem funcionar?

Considere 25 classificadores binários, cada um com taxa de erro \(\epsilon = 0{,}35\). Se eles fossem independentes, o ensemble por voto majoritário erraria apenas quando 13 ou mais classificadores errassem simultaneamente.

Vamos calcular essa probabilidade.

In [ ]:
from math import comb

def majority_vote_error(n_classifiers, epsilon):
    min_errors_for_majority = n_classifiers // 2 + 1
    return sum(
        comb(n_classifiers, i) * (epsilon ** i) * ((1 - epsilon) ** (n_classifiers - i))
        for i in range(min_errors_for_majority, n_classifiers + 1)
    )

majority_vote_error(25, 0.35)

In [ ]:
eps_values = np.linspace(0.01, 0.49, 100)
ensemble_errors = [majority_vote_error(25, eps) for eps in eps_values]

plt.figure(figsize=(7, 5))
plt.plot(eps_values, eps_values, label="Erro individual")
plt.plot(eps_values, ensemble_errors, label="Erro do ensemble independente")
plt.xlabel("Taxa de erro de cada classificador")
plt.ylabel("Taxa de erro")
plt.title("Efeito do voto majoritário com classificadores independentes")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 8. Bagging e amostragem bootstrap

No **Bagging** (*Bootstrap Aggregating*), cada modelo de base é treinado em uma amostra bootstrap do conjunto de treinamento, isto é, uma amostra aleatória **com reposição**.

Em média, cada amostra bootstrap contém cerca de 63% das observações únicas do conjunto original. As observações não selecionadas para uma árvore são chamadas de **out-of-bag** e podem ser usadas para estimar desempenho.

In [ ]:
def bootstrap_sample_indices(n, random_state=None):
    rng = np.random.default_rng(random_state)
    return rng.integers(0, n, size=n)

n = 1000
n_trials = 2000
unique_fractions = []

a = []
for i in range(n_trials):
    idx = bootstrap_sample_indices(n, random_state=i)
    unique_fractions.append(len(np.unique(idx)) / n)

print("Fração média de observações únicas:", np.mean(unique_fractions))
print("Valor teórico aproximado 1 - 1/e:", 1 - 1 / np.e)

plt.figure(figsize=(7, 5))
plt.hist(unique_fractions, bins=30, edgecolor="k")
plt.axvline(1 - 1 / np.e, linestyle="--", label="1 - 1/e")
plt.xlabel("Fração de observações únicas na amostra bootstrap")
plt.ylabel("Frequência")
plt.title("Amostragem bootstrap")
plt.legend()
plt.show()

## 9. Bagging com árvores de decisão

Árvores são modelos instáveis: pequenas mudanças nos dados podem produzir árvores diferentes. Isso as torna boas candidatas para bagging.

In [ ]:
bagging = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
    n_estimators=60,
    bootstrap=True,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
)

try:
    bagging.fit(X_train, y_train)
except TypeError:
    # Compatibilidade com versões antigas do scikit-learn
    bagging = BaggingClassifier(
        base_estimator=DecisionTreeClassifier(random_state=RANDOM_STATE),
        n_estimators=60,
        bootstrap=True,
        oob_score=True,
        random_state=RANDOM_STATE,
        n_jobs=1,
    )
    bagging.fit(X_train, y_train)

print("Acurácia treino:", accuracy_score(y_train, bagging.predict(X_train)))
print("Acurácia teste:", accuracy_score(y_test, bagging.predict(X_test)))
print("OOB score:", bagging.oob_score_)

plot_decision_boundary(bagging, X_test, y_test, "Bagging com árvores de decisão")
plt.show()

## 10. Random Forest

Random Forest combina duas fontes de aleatoriedade:

1. Amostragem bootstrap das observações.
2. Seleção aleatória de subconjuntos de atributos em cada nó.

Isso aumenta a diversidade das árvores e tende a reduzir variância.

In [ ]:
rf = RandomForestClassifier(
    n_estimators=60,
    max_features="sqrt",
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=1,
)
rf.fit(X_train, y_train)

print("Acurácia treino:", accuracy_score(y_train, rf.predict(X_train)))
print("Acurácia teste:", accuracy_score(y_test, rf.predict(X_test)))
print("OOB score:", rf.oob_score_)

plot_decision_boundary(rf, X_test, y_test, "Random Forest")
plt.show()

### Importância de atributos em Random Forest

Vamos usar o dataset de câncer de mama para observar como o modelo estima a importância relativa de atributos.

In [ ]:
cancer = load_breast_cancer()
X_cancer = cancer.data
y_cancer = cancer.target
feature_names = cancer.feature_names

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_cancer, y_cancer, test_size=0.30, random_state=RANDOM_STATE, stratify=y_cancer
)

rf_cancer = RandomForestClassifier(
    n_estimators=60,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=1,
)
rf_cancer.fit(Xc_train, yc_train)

pred = rf_cancer.predict(Xc_test)
print("Acurácia:", accuracy_score(yc_test, pred))
print(classification_report(yc_test, pred, target_names=cancer.target_names))

importances = pd.DataFrame({
    "atributo": feature_names,
    "importancia": rf_cancer.feature_importances_,
}).sort_values("importancia", ascending=False)

importances.head(10)

In [ ]:
top = importances.head(12).iloc[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top["atributo"], top["importancia"])
plt.xlabel("Importância")
plt.title("Random Forest — atributos mais importantes")
plt.tight_layout()
plt.show()

## 11. Boosting: AdaBoost e Gradient Boosting

Em boosting, os modelos são treinados sequencialmente, e cada novo modelo tenta corrigir limitações dos anteriores.

- **AdaBoost:** aumenta o peso de observações classificadas incorretamente e combina modelos por votação ponderada.
- **Gradient Boosting:** treina modelos sucessivos para reduzir a função de custo, frequentemente ajustando novos modelos aos resíduos ou gradientes dos anteriores.

In [ ]:
ada = safe_adaboost(
    DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
    n_estimators=80,
    learning_rate=0.5,
    random_state=RANDOM_STATE,
)

gb = GradientBoostingClassifier(
    n_estimators=80,
    learning_rate=0.05,
    max_depth=2,
    random_state=RANDOM_STATE,
)

boost_results = []
for name, model in [("AdaBoost", ada), ("Gradient Boosting", gb)]:
    boost_results.append(evaluate_classifier(model, X_train, X_test, y_train, y_test, name))

pd.DataFrame(boost_results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_decision_boundary(ada, X_test, y_test, "AdaBoost", ax=axes[0])
plot_decision_boundary(gb, X_test, y_test, "Gradient Boosting", ax=axes[1])
plt.tight_layout()
plt.show()

## 12. Comparação controlada de modelos

Vamos comparar vários modelos no mesmo particionamento do dataset `make_moons`.

In [ ]:
models = {
    "Árvore sem restrição": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Árvore max_depth=3": DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE),
    "Bagging": bagging,
    "Random Forest": RandomForestClassifier(n_estimators=60, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=1),
    "AdaBoost": safe_adaboost(DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE), n_estimators=80, learning_rate=0.5, random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=80, learning_rate=0.05, max_depth=2, random_state=RANDOM_STATE),
}

comparison = []
for name, model in models.items():
    comparison.append(evaluate_classifier(model, X_train, X_test, y_train, y_test, name))

comparison_df = pd.DataFrame(comparison).sort_values("acc_teste", ascending=False).reset_index(drop=True)
comparison_df

## 13. Stacking

No stacking, as predições de modelos de base são usadas como entrada para um metamodelo. Em vez de escolher uma regra fixa de agregação, treinamos um modelo para combinar os classificadores.

In [ ]:
base_estimators = [
    ("tree", DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)),
    ("knn", make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15))),
    ("svc", make_pipeline(StandardScaler(), SVC(C=1.0, gamma="scale", random_state=RANDOM_STATE))),
]

stack = StackingClassifier(
    estimators=base_estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5,
    n_jobs=1,
)

stack.fit(X_train, y_train)
print("Acurácia treino:", accuracy_score(y_train, stack.predict(X_train)))
print("Acurácia teste:", accuracy_score(y_test, stack.predict(X_test)))

plot_decision_boundary(stack, X_test, y_test, "Stacking")
plt.show()

### Exercício 2 — Experimento com Random Forest

Use o dataset de câncer de mama e investigue o efeito dos hiperparâmetros:

- `n_estimators`: 50, 100, 300.
- `max_depth`: `None`, 3, 5.
- `max_features`: `"sqrt"`, `"log2"`.

Monte uma tabela com a média de acurácia em validação cruzada estratificada com 5 folds. Depois, indique a melhor combinação.

In [ ]:
# TODO: implemente o experimento descrito no enunciado.
# Dicas:
# - use StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
# - use cross_val_score(..., scoring="accuracy")
# - armazene os resultados em uma lista de dicionários e converta para DataFrame

rf_grid_results = pd.DataFrame()
rf_grid_results

### Exercício 3 — Bagging, Random Forest e Boosting em dados ruidosos

Crie um dataset sintético com `make_classification` e compare:

1. Uma árvore sem restrição.
2. Bagging com árvores.
3. Random Forest.
4. AdaBoost.
5. Gradient Boosting.

Use acurácia em treino e validação (use cross-validation). Em seguida, responda: qual modelo parece generalizar melhor? Há sinais de sobreajuste? Faça uma avaliação final com o melhor modelo no conjunto de teste.

In [ ]:
# TODO: crie o dataset sintético com make_classification, faça a divisão treino/teste
# e compare os modelos listados no enunciado.

syn_results = pd.DataFrame()
syn_results

# 14. Mini-projeto final

Escolha um dos caminhos:

**Caminho A — Classificação:** use o dataset de câncer de mama (`load_breast_cancer`).

**Caminho B — Regressão:** use o dataset diabetes (`load_diabetes`).

Tarefas:

1. Defina uma métrica principal.
2. Compare ao menos três modelos, incluindo uma árvore individual e dois ensembles.
3. Faça um pequeno ajuste de hiperparâmetros.
4. Interprete os resultados: desempenho, sobreajuste, interpretabilidade e custo computacional.

A célula abaixo implementa uma solução de referência para o Caminho A.

In [ ]:
# TODO: implemente seu mini-projeto aqui.
# Escolha classificação ou regressão e compare pelo menos 3 modelos.
# Inclua uma árvore individual e dois ensembles.

# Seu código:

# 15. Perguntas de consolidação

1. Por que árvores profundas tendem a sobreajustar?
2. Qual a diferença entre pré-poda e pós-poda?
3. Por que bagging é especialmente útil com árvores de decisão?
4. Qual a diferença entre Bagging e Random Forest?
5. Por que boosting é sequencial?
6. Em que situação stacking pode ser mais interessante do que votação simples?
7. Qual modelo desta aula você escolheria para um problema em que interpretabilidade é essencial? E para um problema em que desempenho preditivo é prioridade?